# Unconditional 2D Gaussian mixture (continual learning)

Two well-separated Gaussian modes in $\mathbb{R}^2$. Each mode is a **task**.
Labels are used only to split pretrain / finetune data; generation is **unconditional**.

Sampler: `trajectory_replay.data.mixture_mog`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_repo = Path.cwd()
while _repo != _repo.parent:
    if (_repo / "pyproject.toml").is_file() and (_repo / "src" / "trajectory_replay").is_dir():
        break
    _repo = _repo.parent
else:
    raise FileNotFoundError("Could not locate repo root (pyproject.toml) from this notebook path.")

sys.path.insert(0, str((_repo / "src").resolve()))

from trajectory_replay.data.mixture_mog import (
    GaussianMixtureDataset,
    make_two_mode_means,
    sample_gaussian_mixture,
    sample_mixture_with_weights,
)

rng = np.random.default_rng(0)

In [ ]:
def scatter_by_task(ds: GaussianMixtureDataset, title: str) -> None:
    fig, ax = plt.subplots(figsize=(6, 5))
    cmap = plt.get_cmap("tab10")
    for k in range(ds.n_tasks):
        m = ds.mask_task(k)
        ax.scatter(
            ds.x[m, 0],
            ds.x[m, 1],
            s=16,
            alpha=0.55,
            color=cmap(k % 10),
            edgecolors="none",
            label=f"task {k}",
        )
        ax.scatter(
            ds.means[k, 0],
            ds.means[k, 1],
            s=120,
            marker="X",
            color=cmap(k % 10),
            edgecolors="k",
            linewidths=0.8,
            zorder=5,
        )
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()


## Two-task MoG

Means at $(\pm s/2, 0)$ with isotropic noise $\sigma$. Task 0 = pretrain, task 1 = finetune.

In [ ]:
means = make_two_mode_means(dim=2, separation=3.0, axis=0)
print("means:\n", means)

ds = sample_gaussian_mixture(
    means,
    samples_per_task=10000,
    rng=rng,
    sigma=0.5
)
print(f"x={ds.x.shape}, tasks={np.bincount(ds.task)}")

scatter_by_task(ds, r"Samples: 2-mode MoG, $\sigma=0.35$")

## Pretrain / finetune split

Unconditional training uses only `x`; `task` selects which component is available at each stage.

In [ ]:
ds_pretrain = ds.subset(0)
ds_finetune = ds.subset(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, subset, title in [
    (axes[0], ds_pretrain, "Pretrain: task 0 only"),
    (axes[1], ds_finetune, "Finetune: task 1 only"),
]:
    ax.scatter(subset.x[:, 0], subset.x[:, 1], s=16, alpha=0.55, c="C0", edgecolors="none")
    for k in range(ds.n_tasks):
        ax.scatter(
            ds.means[k, 0],
            ds.means[k, 1],
            s=120,
            marker="X",
            c=f"C{k}",
            edgecolors="k",
            linewidths=0.8,
            zorder=5,
            label=f"$\\mu_{k}$",
        )
    # shared axes require adjustable='box' (datalim is illegal with sharex+sharey)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.legend(loc="best")
axes[0].set_ylabel("$x_2$")
plt.tight_layout()
plt.show()

## Target mixture $\tilde{q} = \lambda q + (1-\lambda) q'$

Reference samples for mixture-recovery checks (same means / $\sigma$, weighted component draws).

In [ ]:
lam = 0.7
ds_mix = sample_mixture_with_weights(
    means,
    weights=np.array([lam, 1.0 - lam]),
    n_samples=10000,
    rng=np.random.default_rng(1),
    sigma=0.5
)
emp = np.bincount(ds_mix.task, minlength=2) / len(ds_mix.task)
print(f"target weights = ({lam:.2f}, {1 - lam:.2f})")
print(f"empirical masses = ({emp[0]:.3f}, {emp[1]:.3f})")

scatter_by_task(ds_mix, rf"Mixture samples: $\lambda={lam}$")